# 01 — Synthetic corpus generation (conditions D, E, E′)

Generates the three 6,000-pair Q&A corpora with GPT-5 Mini via `scripts/generate_corpus.py`.

| mode | condition | output file |
|---|---|---|
| `meta` | D — alignment pretraining as hyperstition | `meta_awareness_corpus.jsonl` |
| `benign` | E — general science control | `benign_control_corpus.jsonl` |
| `ai_control` | E′ — AI training/alignment, engineering framing | `ai_control_corpus.jsonl` |

Each mode: six topic clusters × five registers, 80–135 tokens per pair (GPT-NeoX tokenizer), deduplicated, resumable. Each run writes a `*_stats.json` alongside the corpus.

**Before running:** upload `generate_corpus.py` to `/content/` (or change the path in the cells).

## Setup

In [ ]:
# Cell 1: mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')
!pip install openai transformers --quiet

In [2]:
# API key — read from the Colab secrets panel (key icon in the left sidebar).
# Add a secret named OPENAI_API_KEY there; never paste the key into a cell.
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


## Test runs (~60 pairs each, written to separate `_TEST` files)

In [ ]:
# Cell 3: TEST RUN - meta awareness corpus (~60 pairs)
!python /content/generate_corpus.py --mode meta --test \
    --output-dir /content/drive/MyDrive/experiment/synthetic_data

In [ ]:
# Cell 4: TEST RUN - benign control corpus (~60 pairs)
!python /content/generate_corpus.py --mode benign --test \
    --output-dir /content/drive/MyDrive/experiment/synthetic_data

In [ ]:
# Cell 4a: test run - AI control corpus
!python /content/generate_corpus.py --mode ai_control --test --output-dir /content/drive/MyDrive/experiment/synthetic_data

## Full runs (6,000 pairs each)

Outputs below are the generation logs from the runs used in the experiment. The meta-awareness log is truncated by Colab's output limit; its final counts are in `meta_awareness_stats.json`.

In [ ]:
# Cell 5: meta-awareness corpus  (~10-15 min)
!python /content/generate_corpus.py --mode meta \
    --output-dir /content/drive/MyDrive/experiment/synthetic_data

In [ ]:
# Cell 6: benign control corpus (~10-15 min)
!python /content/generate_corpus.py --mode benign \
    --output-dir /content/drive/MyDrive/experiment/synthetic_data

In [ ]:
# Cell 7: AI Control corpus full run (~10-15 min)
!python /content/generate_corpus.py --mode ai_control --output-dir /content/drive/MyDrive/experiment/synthetic_data

## One-off repair applied to the E corpus

GPT-5 Mini occasionally adds a stray top-level `"role"` key beside `"messages"`. This cell stripped it from the benign corpus file in place (10 pairs). `load_corpus()` in the fine-tuning script also ignores stray keys.

In [ ]:
# Quick fix — run in a Colab cell
import json

path = "/content/drive/MyDrive/experiment/synthetic_data/benign_control_corpus.jsonl"
lines = open(path).readlines()
fixed = 0
with open(path, "w") as f:
    for line in lines:
        obj = json.loads(line)
        if "role" in obj and "messages" in obj:
            del obj["role"]
            fixed += 1
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
print(f"Stripped stray top-level 'role' from {fixed} pairs")